# CAFE — Mode B: composed pipelines (techniques)

Instead of a sealed black box, build your system out of **techniques** and call them
through an instrumented context `ctx`. CAFE then:

- **swaps techniques as factors** (a factor per *stage*, levels = your registered techniques),
- records **per-stage timing/cost** and a **trace** of every run,
- runs the same judging + statistics as Mode A.

No fixed stages — *you* name the stages and write the topology in plain Python. Here:
a 2-stage system **answer → refine**, on TruthfulQA, via Ollama Cloud.

In [1]:
import cafe
from cafe._env import load_env
from cafe.techniques import registry

load_env()
registry.clear()  # so re-running the notebook doesn't double-register
print("cafe", cafe.__version__)

cafe 0.0.1


## 1. Register techniques

Two **stages** (your names): `answer` and `refine`. Each has two techniques. A
keyword arg with a default (none here) would become a tunable parameter factor.

In [2]:
MODEL = "ollama_cloud/gpt-oss:120b"

@cafe.technique("answer", "direct")
async def answer_direct(ctx, question):
    return await cafe.complete(MODEL, [
        {"role": "system", "content": "Answer truthfully and concisely."},
        {"role": "user", "content": question}])

@cafe.technique("answer", "cot")
async def answer_cot(ctx, question):
    return await cafe.complete(MODEL, [
        {"role": "system", "content": "Reason step by step about common misconceptions, then give the truthful answer."},
        {"role": "user", "content": question}])

@cafe.technique("refine", "none")
async def refine_none(ctx, question, draft):
    return draft  # no LLM call -> ~free, ~instant

@cafe.technique("refine", "critique")
async def refine_critique(ctx, question, draft):
    return await cafe.complete(MODEL, [
        {"role": "system", "content": "Critique the draft answer for truthfulness and fix any errors. Output only the improved answer."},
        {"role": "user", "content": f"Question: {question}\n\nDraft: {draft}"}])

print("stages:", cafe.techniques.stages(), "| answer techniques:", cafe.techniques.names_for("answer"))

stages: ['answer', 'refine'] | answer techniques: ['direct', 'cot']


## 2. Compose them into a system

The topology is just your Python: answer, then refine. CAFE watches each `ctx.run`.

In [3]:
async def my_system(config, item, ctx):
    draft = await ctx.run("answer", question=item["text"])
    final = await ctx.run("refine", question=item["text"], draft=draft)
    return final

## 3. Build the study

`cafe.technique_factor(stage)` auto-derives the factor levels from the registry, so
you don't hand-list them. Wrap the system with `cafe.composed(...)`.

In [4]:
study = cafe.Study(
    name="rag-modeB",
    system=cafe.composed(my_system),
    factors=[
        cafe.technique_factor("answer"),   # levels: direct, cot
        cafe.technique_factor("refine"),   # levels: none, critique
    ],
    dataset=cafe.datasets.load_truthfulqa(n=5, categories=["Misconceptions"], seed=1),
    rubric=cafe.ANSWER_QUALITY_1_5,
    judge=cafe.LLMJudge(model=MODEL),
    replications=1,
)
print(cafe.size(study), "configs:", cafe.generate(study))

4 configs: [{'answer': 'direct', 'refine': 'none'}, {'answer': 'direct', 'refine': 'critique'}, {'answer': 'cot', 'refine': 'none'}, {'answer': 'cot', 'refine': 'critique'}]


In [5]:
result = await cafe.evaluate(study, concurrency=4)
result

rag-modeB: answers:   0%|          | 0/20 [00:00<?, ?it/s]

judging:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluation(20 answers · 4 configs · 5 inputs · 20 ratings · best: answer=cot·refine=critique)

## 4. Per-stage view (the Mode B payoff)

Because every `ctx.run` was timed, we can see where time goes — `refine/none` is
~instant (no LLM call), `refine/critique` adds a real second model call.

In [6]:
import pandas as pd
pd.DataFrame(cafe.stage_report(result.answers))

,stage,technique,calls,mean_elapsed_s,mean_cost_usd
0,refine,critique,10,12.4986,0.0
1,answer,cot,10,11.5625,0.0
2,answer,direct,10,4.5305,0.0
3,refine,none,10,0.0000,0.0


### The trace of one run

Every answer records the exact sequence of techniques it went through.

In [7]:
for step in result.answers.observations[-1].metadata["trace"]:
    print(f"  {step['stage']:>8} / {step['technique']:<8}  {step['elapsed_s']:.2f}s")

    answer / cot       12.62s
    refine / critique  15.33s


## 5. Statistics — same as Mode A

Which stage's choice (answer strategy vs refine) drove quality?

In [8]:
print(result.attribution.show())
print()
print(result.effects.show())

ratings: 20 (20 usable)   factors: answer, refine

per-configuration mean quality:
  5.00  (n= 5)  answer=cot·refine=critique
  5.00  (n= 5)  answer=cot·refine=none
  5.00  (n= 5)  answer=direct·refine=critique
  4.60  (n= 5)  answer=direct·refine=none

per-factor marginal means:
  answer:
     cot              mean=5.00  n=10
     direct           mean=4.80  n=10
  refine:
     critique         mean=5.00  n=10
     none             mean=4.80  n=10

best configuration: answer=cot·refine=critique  (mean 5.00)

model: one-way ANOVA (mixed model failed)   (n=20, α=0.05)

factor effects (is it real? how much variance?):
  factor                  F         p   partial η²   significant
  refine               1.00    0.3306        0.053     no
  answer               1.00    0.3306        0.053     no

pairwise effect sizes (Cohen's d):
  answer: cot vs direct  d=+0.45  [-0.44, +1.33]
  refine: critique vs none  d=+0.45  [-0.44, +1.33]
  ! mixed model did not fit (Singular matrix); using one-w

/home/fabian/Desktop/PHD/CAFE/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/fabian/Desktop/PHD/CAFE/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/home/fabian/Desktop/PHD/CAFE/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/fabian/Desktop/PHD/CAFE/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)


## Notes

- **Stages are yours** — `answer`/`refine` are names you chose, not fixed slots. Add
  branches, cascades, loops; the topology is your Python.
- Tunable technique params (a kwarg with a default) become parameter factors
  (`answer.temperature`), swept alongside the technique choice.
- The same trace/cost flows into the platform's run-inspector later; here it's just
  metadata on each answer (`obs.metadata['trace']`).